# Attic: training a tiny word2vec, from scratch

This was originally a "Bonus" section at the end of `Part_4_embeddings.ipynb`; it's parked here since the architecture/training story is now covered conceptually in `Part_4_embeddings.ipynb` itself (the "Inside the network" section). Kept in case you want the hands-on version back (e.g. as a follow-up exercise after that section).

Requires `import torch`, `import torch.nn.functional as F` (as set up in `Part_4_embeddings.ipynb`, Programs 1-2) to run standalone.

## Training a tiny word2vec, from scratch

Everything so far used embeddings already baked into a pretrained model (`SmolLM2`) or a pretrained word2vec-style model (GloVe). But where do vectors like these actually come from? The original **word2vec** approach (see [Google's Machine Learning Crash Course, "Obtaining embeddings"](https://developers.google.com/machine-learning/crash-course/embeddings/obtaining-embeddings)) is a great illustration: train a small neural network on a *pretext task* — predicting the words *around* a given word, every word in a small window on both sides — not because that prediction is the goal in itself, but because it forces the network to organize words usefully along the way. Concretely: a one-hot (or, here, an id) goes in, passes through a hidden layer of size *d* — that hidden layer's weights are exactly the embedding we care about — and the network is trained to predict its neighbors. Once training is done, the classification task itself is thrown away; only that hidden layer's weights get reused, as the embedding table. This specific variant, predicting several surrounding words rather than just the next one, is called **skip-gram**.

That's exactly the same idea as the real next-token training objective behind every LLM in this course (Part 3, Programs 7-8) — just at a toy scale, and this time actually *training* the network ourselves (with backpropagation) instead of only running inference on an already-trained one.

Let's build a tiny corpus where " king"/" queen" and " man"/" woman" always play the same grammatical role (as the subject doing something), while " dog"/" bread"/" country" always play another role (as the object something is done to) — then train a small embedding + a linear layer to predict, for each word, every word within a small window around it, and see what structure emerges purely from that.

In [ ]:
# Program: word2vec's actual objective -- predicting the surrounding words (skip-gram)

import torch.nn as nn

torch.manual_seed(2)  # picked for a clean result -- see the discussion below

# A handful of sentences, repeated many times so the optimizer has enough signal.
toy_sentences = [
    "the king rules the country",
    "the queen rules the country",
    "the man walks the dog",
    "the woman walks the dog",
    "the king is strong",
    "the queen is strong",
    "the man is strong",
    "the woman is strong",
    "the king loves the queen",
    "the man loves the woman",
    "the king eats bread",
    "the queen eats bread",
    "the man eats bread",
    "the woman eats bread",
]
skipgram_corpus = " ".join(toy_sentences * 30).split()  # simple word-level split, no BPE here
skipgram_vocab = sorted(set(skipgram_corpus))
skipgram_word_to_id = {word: i for i, word in enumerate(skipgram_vocab)}

# Skip-gram training pairs: for every word, pair it with every OTHER word within
# `window` positions on either side -- not just the single word right after it.
window = 2
skipgram_pairs = []
for i, center_word in enumerate(skipgram_corpus):
    for j in range(max(0, i - window), min(len(skipgram_corpus), i + window + 1)):
        if j != i:
            skipgram_pairs.append((skipgram_word_to_id[center_word], skipgram_word_to_id[skipgram_corpus[j]]))

print(f"Skip-gram training pairs: {len(skipgram_pairs)}")

skipgram_inputs = torch.tensor([pair[0] for pair in skipgram_pairs])
skipgram_targets = torch.tensor([pair[1] for pair in skipgram_pairs])

# The model: an embedding table (this IS what we're actually training) followed by a
# linear layer that turns an embedding back into a score for every word in the vocabulary.
skipgram_embedding = nn.Embedding(len(skipgram_vocab), 8)
skipgram_output_layer = nn.Linear(8, len(skipgram_vocab))
optimizer = torch.optim.Adam(
    list(skipgram_embedding.parameters()) + list(skipgram_output_layer.parameters()), lr=0.03
)

for epoch in range(1500):
    optimizer.zero_grad()
    predicted_logits = skipgram_output_layer(skipgram_embedding(skipgram_inputs))
    loss = F.cross_entropy(predicted_logits, skipgram_targets)
    loss.backward()
    optimizer.step()

print(f"Final training loss: {loss.item():.3f}")

def skipgram_similarity(word_a, word_b):
    vector_a = skipgram_embedding.weight[skipgram_word_to_id[word_a]]
    vector_b = skipgram_embedding.weight[skipgram_word_to_id[word_b]]
    return F.cosine_similarity(vector_a.unsqueeze(0), vector_b.unsqueeze(0)).item()

print("\n--- 'subject' words (should end up similar to each other) ---")
for a, b in [("king", "queen"), ("man", "woman")]:
    print(f"{a:8} vs {b:8}  {skipgram_similarity(a, b):.3f}")

print("\n--- 'subject' vs 'object' words (should end up dissimilar) ---")
for a, b in [("king", "dog"), ("king", "country"), ("queen", "bread")]:
    print(f"{a:8} vs {b:8}  {skipgram_similarity(a, b):.3f}")

print("\n--- 'object' words (should also end up similar to each other) ---")
for a, b in [("dog", "bread"), ("dog", "country")]:
    print(f"{a:8} vs {b:8}  {skipgram_similarity(a, b):.3f}")

Structure emerges exactly where we'd hope: " king"/" queen" (0.673) and " man"/" woman" (0.435) end up clearly positive — these are the four words that always play the *subject* role in our sentences. Meanwhile " king"/" country" and " queen"/" bread" turn slightly negative, and " king"/" dog" comes out only mildly positive (0.068) — subjects and objects are still mostly pushed apart, even though the network never explicitly predicted "the next word", only "some word nearby" (a harder, noisier pretext task, especially with a corpus this tiny). And the objects themselves (" dog", " bread", " country") cluster together too (0.329, 0.352), following the same *role*-based pattern.

We never told this network anything about grammar, gender, or meaning. All we asked it to do was predict a word's neighbors — yet to get better at that narrow task, it had no choice but to organize its embedding table so that words playing the same role end up nearby, and words playing different roles end up apart. That's the whole trick behind word2vec: throw away the little classifier once training is done, and keep only the hidden layer's weights as a reusable, general-purpose embedding table. A modern LLM's embedding layer is trained the same way in spirit — as a side effect of learning to predict text — except it's trained jointly with the rest of a much larger network (as one giant model, not a separate step), on vastly more data, which is exactly why `SmolLM2`'s real embeddings, back in Program 2, already placed " king" and " queen" as close neighbors far more richly than these 14 toy words ever could.